In [1]:
# Imports
import json
import pandas as pd
from datasets import load_dataset

/opt/anaconda3/envs/llm-risk-control/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_balanced_dataset(df):
    # Balance the dataset
    balanced = []
    n = df['label'].value_counts().min()
    for label, group in df.groupby('label'):
        group = group.sample(frac=1)[:n]
        balanced.append(group)
    
    return pd.concat(balanced).reset_index(drop=True)

In [3]:
unprocessed_data_dir = "./datasets/icl/"
data_output_dir = "./processed_data/icl/"

# Text-based Classification

In [ ]:
# AG News (AG_News)
df = pd.read_csv(unprocessed_data_dir + 'ag_news/test.csv')
# Append the title and description to get the text
df['text'] = [t + ": " + d for t,d in zip(df['Title'], df['Description'])]
df['label'] = df['Class Index'].replace(1, 'world').replace(2, 'sports').replace(3, 'business')
df['label'] = df['label'].replace(4, 'science/technology')
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'ag_news.csv', index=False)

In [55]:
# Text Retrieval Conference (TREC)
# https://huggingface.co/datasets/CogComp/trec
df = pd.read_csv(unprocessed_data_dir + 'trec/train.csv')
df['label'] = df['label-coarse'].replace(0, 'abbreviation').replace(1, 'entity').replace(2, 'description')
df['label'] = df['label'].replace(3, 'human').replace(4, 'location').replace(5, 'numeric')
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'trec.csv', index=False)

In [64]:
# Spam detection (SpamAssassin)
# no = not spam, yes = spam
df = pd.read_csv(unprocessed_data_dir + 'spam/spam_assassin.csv')
df['label'] = df['target'].replace(0, 'no').replace(1, 'yes')
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'spam.csv', index=False)

# Paraphrase & Entailment

In [6]:
# Medical Question Pairs (MQP)
# Medical Question Pairs dataset by McCreery et al (2020) contains pairs of medical questions and paraphrased versions 
# of the question prepared by medical professional. Paraphrased versions were labelled as similar (syntactically dissimilar 
# but contextually similar) or dissimilar (syntactically may look similar but contextually dissimilar). 
# Labels 1: similar, 0: dissimilar
ds = load_dataset("bigbio/mqp", "mqp_source")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)
df['text'] = ['Text 1: ' + t1 + ' Text 2: ' + t2 for t1,t2 in zip(df['text_1'], df['text_2'])]
df['label'] = df['label'].replace('0', 'dissimilar').replace('1', 'similar')
df = df[['text', 'label']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'mqp.csv', index=False)

In [7]:
# Microsoft Research Paraphrase Corpus (MRP)
# labels: 1 - equivalent, 0 - different
ds = load_dataset('glue', 'mrpc', split='train')
df = pd.DataFrame(data=ds, columns=ds.features)
df['text'] = ['Sentence 1: ' + t1 + ' Sentence 2: ' + t2 for t1,t2 in zip(df['sentence1'], df['sentence2'])]
df['label'] = df['label'].replace(0, 'different').replace(1, 'equivalent')
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'mrp.csv', index=False)

In [8]:
# Winograd NLI (WNLI)
ds = load_dataset("SetFit/wnli")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)
df['text'] = ['Text 1: ' + t1 + ' Text 2: ' + t2 for t1,t2 in zip(df['text1'], df['text2'])]
df['label'] = df['label'].replace(0, 'is entailment').replace(1, 'not entailment')
df = df[['text', 'label']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'wnli.csv', index=False)

Repo card metadata block was not found. Setting CardData to empty.


# Sentiment Analysis

In [10]:
# FinancialPhrasebank
df = pd.read_csv(unprocessed_data_dir + 'financial_phrasebank/financial_phrasebank_processed.csv')
df['label'] = df['Sentiment']
df['text'] = df['Text']
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'financial_phrasebank.csv', index=False)

In [11]:
# SST2
ds = load_dataset("SetFit/sst2")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)[['text', 'label_text']] 
df['label'] = df['label_text']
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'sst2.csv', index=False)

Repo card metadata block was not found. Setting CardData to empty.


In [12]:
# TweetEval-Atheism
ds = load_dataset("cardiffnlp/tweet_eval", "stance_atheism")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)[['text', 'label']] 
df['label'] = df['label'].replace(0, 'neither').replace(1, 'no').replace(2, 'yes')
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'tweeteval_atheism.csv', index=False)

In [13]:
# TweetEval-Feminist
ds = load_dataset("cardiffnlp/tweet_eval", "stance_feminist")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)[['text', 'label']] 
df['label'] = df['label'].replace(0, 'neither').replace(1, 'no').replace(2, 'yes')
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'tweeteval_feminist.csv', index=False)

In [14]:
# TweetEval-Hate
ds = load_dataset("cardiffnlp/tweet_eval", "hate")
df = pd.DataFrame(data=ds['train'], columns=ds['train'].features)[['text', 'label']] 
df['label'] = df['label'].replace(0, 'favor').replace(1, 'against')
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'tweeteval_hate.csv', index=False)

In [15]:
# Unnatural
df = pd.read_csv(unprocessed_data_dir + 'unnatural/unnatural.csv')
df = df[['label', 'text']]
df = get_balanced_dataset(df)
df.to_csv(data_output_dir + 'unnatural.csv', index=False)

# BigBench-Hard

In [17]:
def get_balanced_bbh_dataset(data):
    text = [item['input'] for item in data['examples']]
    labels = [item['target'].lower() for item in data['examples']]
    
    df = pd.DataFrame()
    df['text'] = text
    df['label'] = labels
    return get_balanced_dataset(df)

In [18]:
# Boolean
with open(unprocessed_data_dir + 'boolean/boolean_expressions.json', 'r') as file:
    data = json.load(file)

df_balanced = get_balanced_bbh_dataset(data)
df_balanced.to_csv(data_output_dir + 'boolean.csv', index=False)

In [19]:
# Navigation
with open(unprocessed_data_dir + 'navigation/navigate.json', 'r') as file:
    data = json.load(file)

df_balanced = get_balanced_bbh_dataset(data)
df_balanced.to_csv(data_output_dir + 'navigation.csv', index=False)

In [20]:
# Sports Understanding
with open(unprocessed_data_dir + 'sports/sports_understanding.json', 'r') as file:
    data = json.load(file)

df_balanced = get_balanced_bbh_dataset(data)
df_balanced.to_csv(data_output_dir + 'sports.csv', index=False)

In [21]:
# Web of Lies
with open(unprocessed_data_dir + 'web_of_lies/web_of_lies.json', 'r') as file:
    data = json.load(file)

df_balanced = get_balanced_bbh_dataset(data)
df_balanced.to_csv(data_output_dir + 'web_of_lies.csv', index=False)